In [1]:
import time
notebook_start = time.perf_counter()

import os, json, pandas as pd, numpy as np, joblib, shap
import kditransform
from thermoift import print_model_metrics
from thermoift.rng_utils import get_rng
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.inspection import permutation_importance
from tabpfn_extensions.interpretability.shap import get_shap_values

In [2]:
PLOT_FOLDER = "TabPFN_interfacial_thickness_OUTPUTS"
target      = "interfacial_thickness"
SEED        = 655552
TEST_ROWS   = None
N_TRIALS   = 100

In [3]:
# Parameters
PLOT_FOLDER = "/gpfs/home6/draju/A6/TabPFN/with_HPO/SLURMTHICKNESS"
TEST_ROWS = None
SEED = 655552
N_TRIALS = 100


In [4]:
import warnings
warnings.filterwarnings(
    "ignore",
    message="TabPFN fit/predict failed at leaf",
    category=UserWarning,
    module="tabpfn_extensions",
)
os.environ["TABPFN_ALLOW_CPU_LARGE_DATASET"] = "1"

try:
    import tabpfn
    from tabpfn import TabPFNRegressor
    from tabpfn.constants import ModelVersion
    from tabpfn_extensions.hpo import TunedTabPFNRegressor

    print(f"TabPFN version: {tabpfn.__version__}")
    print("Selected model version:", ModelVersion.V2)

except ImportError as exc:
    raise ImportError("tabpfn is not installed in this Python environment.") from exc

n_cpus = int(os.environ.get("SLURM_CPUS_PER_TASK", 4))
os.environ["OMP_NUM_THREADS"]      = str(n_cpus)
os.environ["MKL_NUM_THREADS"]      = str(n_cpus)
os.environ["OPENBLAS_NUM_THREADS"] = str(n_cpus)
os.environ["NUMEXPR_NUM_THREADS"]  = str(n_cpus)
try:
    import torch
    torch.set_num_threads(n_cpus)
except ImportError:
    pass
print(f"Thread limit set to {n_cpus} (SLURM_CPUS_PER_TASK)")

TabPFN version: 7.1.1
Selected model version: ModelVersion.V2
Thread limit set to 16 (SLURM_CPUS_PER_TASK)


In [5]:
df = pd.read_csv("../../DATASET_A4/interfacial_results_dataset_A4.csv")
print(f"Number of rows: {len(df)}")

if isinstance(TEST_ROWS, str) and TEST_ROWS.strip().lower() in ("", "none", "null"):
    TEST_ROWS = None
if TEST_ROWS is not None:
    TEST_ROWS = int(TEST_ROWS)
    df = df.iloc[:TEST_ROWS].copy()
    print(f"Test mode: using first {TEST_ROWS} rows only")
else:
    print("Full mode: using all rows")

print(f"Total samples: {len(df)}")
print(f"\n{target} statistics:")
print(df[target].describe())

Number of rows: 19361
Full mode: using all rows
Total samples: 19361

interfacial_thickness statistics:
count    19361.000000
mean         1.353666
std          0.682266
min          0.682573
25%          0.887208
50%          1.109909
75%          1.574425
max          5.659519
Name: interfacial_thickness, dtype: float64


In [6]:
rng       = get_rng(seed=SEED)

z_columns  = [col for col in df.columns if col.startswith("z_")]
Z_non_zero = [col for col in z_columns if (df[col] != 0).any()]
features   = ["temperature", "pressure"] + Z_non_zero

print(f"Selected features: {features}")

X     = df[features]
y     = df[target]
# 70/15/15 split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=SEED)
X_test,  X_val,  y_test,  y_val  = train_test_split(X_temp, y_temp, test_size=0.50, random_state=SEED)

print(f"\nTraining samples:   {X_train.shape[0]}")
print(f"Testing samples:    {X_test.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")

Selected features: ['temperature', 'pressure', 'z_carbon dioxide', 'z_hydrogen', 'z_nitrogen', 'z_argon', 'z_methane', 'z_oxygen', 'z_carbon monoxide', 'z_hydrogen sulfide']

Training samples:   13552
Testing samples:    2904
Validation samples: 2905


In [7]:
# Train TabPFN model with HPO (100 Bayesian-opt trials)
from tabpfn_extensions.hpo.search_space import get_param_grid_hyperopt
from tabpfn.constants import ModelVersion

_search_space = get_param_grid_hyperopt("regression", model_version=ModelVersion.V2_5)
_search_space["ignore_pretraining_limits"] = True

tuned_model = TunedTabPFNRegressor(
    n_trials=N_TRIALS,
    metric="rmse",
    n_validation_size=0.2,
    shuffle_data=True,
    search_algorithm_type="tpe",
    device="auto",
    random_state=SEED,
    verbose=False,
    search_space=_search_space,
)
tuned_model.fit(X_train, y_train)

y_train_pred = tuned_model.predict(X_train)
y_test_pred  = tuned_model.predict(X_test)
y_val_pred   = tuned_model.predict(X_val)

metrics = print_model_metrics(y_train, y_train_pred, y_test, y_test_pred, target, unit="nm", y_val=y_val, y_val_pred=y_val_pred)

Model Performance for interfacial_thickness

Training Set:
  R²:   0.999928
  RMSE: 0.005813 nm
  MAE:  0.000843 nm

Test Set:
  R²:   0.999896
  RMSE: 0.007017 nm
  MAE:  0.001071 nm

Validation Set:
  R²:   0.999909
  RMSE: 0.006418 nm
  MAE:  0.001035 nm


In [8]:
print("=" * 60)
print(f"Best CV {tuned_model.metric.value.upper()}: {tuned_model.best_score_:.6f}")
print("Best hyperparameter configuration:")
for k, v in tuned_model.best_params_.items():
    print(f"  {k}: {v}")
print("=" * 60)

trials_records = []
for t in tuned_model.trials_.trials:
    if t["result"].get("status") != "ok":
        continue
    row = {"trial_id": t["tid"], "loss": t["result"]["loss"]}
    for param, vals in t["misc"]["vals"].items():
        row[param] = vals[0] if len(vals) else None
    trials_records.append(row)

trials_df = pd.DataFrame(trials_records).sort_values("loss").reset_index(drop=True)
print("\nTop 10 trials:")
print(trials_df.head(10))

os.makedirs(PLOT_FOLDER, exist_ok=True)
trials_df.to_csv(os.path.join(PLOT_FOLDER, f"TabPFN_{target}_hpo_trials.csv"), index=False)

with open(os.path.join(PLOT_FOLDER, f"TabPFN_{target}_best_config.json"), "w") as f:
    json.dump({
        "best_score": float(tuned_model.best_score_),
        "metric": tuned_model.metric.value,
        "best_config": {k: (v.item() if hasattr(v, "item") else v) for k, v in tuned_model.best_params_.items()},
    }, f, indent=2, default=str)

print(f"Saved best config + trial history to {PLOT_FOLDER}/")

Best CV RMSE: -0.004257
Best hyperparameter configuration:
  FINGERPRINT_FEATURE: 0
  MIN_UNIQUE_FOR_NUMERICAL_FEATURES: 2
  OUTLIER_REMOVAL_STD: 2
  POLYNOMIAL_FEATURES: 0
  PREPROCESS_TRANSFORMS: 163
  REGRESSION_Y_PREPROCESS_TRANSFORMS: 2
  average_before_softmax: 0
  max_depth: 1
  model_path: 5
  model_type: 0
  n_estimators: 0
  softmax_temperature: 2

Top 10 trials:
   trial_id      loss  FINGERPRINT_FEATURE  MIN_UNIQUE_FOR_NUMERICAL_FEATURES  \
0        53  0.004257                    0                                  2   
1        70  0.004283                    0                                  2   
2        68  0.004337                    0                                  2   
3        75  0.004371                    1                                  0   
4        27  0.004502                    0                                  2   
5        29  0.004530                    0                                  2   
6        66  0.004559                    0               

In [9]:
results_df = pd.DataFrame({
    "idx":       np.concatenate([y_train.index, y_test.index, y_val.index]),
    "actual":    np.concatenate([y_train.values, y_test.values, y_val.values]),
    "predicted": np.concatenate([y_train_pred,  y_test_pred,  y_val_pred]),
    "split":     ["train"]*len(y_train) + ["test"]*len(y_test) + ["val"]*len(y_val),
})

os.makedirs(PLOT_FOLDER, exist_ok=True)
results_df.to_csv(os.path.join(PLOT_FOLDER, f"TabPFN_{target}_predictions.csv"), index=False)
print(f"Predictions saved: {len(results_df)} rows")

best_tabpfn = tuned_model.best_model_
model_path  = os.path.join(PLOT_FOLDER, f"TabPFN_{target}_model.joblib")
joblib.dump(best_tabpfn, model_path)
print(f"Model saved to: {model_path}")

tuner_path = os.path.join(PLOT_FOLDER, f"TabPFN_{target}_tuner.joblib")
joblib.dump(tuned_model, tuner_path)
print(f"Tuner saved to: {tuner_path}")

Predictions saved: 19361 rows
Model saved to: /gpfs/home6/draju/A6/TabPFN/with_HPO/SLURMTHICKNESS/TabPFN_interfacial_thickness_model.joblib


Tuner saved to: /gpfs/home6/draju/A6/TabPFN/with_HPO/SLURMTHICKNESS/TabPFN_interfacial_thickness_tuner.joblib


In [10]:
best_tabpfn = tuned_model.best_model_
cv_results = cross_validate(
    best_tabpfn, X, y, cv=5,
    scoring={
        "r2":   "r2",
        "rmse": "neg_root_mean_squared_error",
        "mae":  "neg_mean_absolute_error",
    },
    n_jobs=n_cpus,
)

cv_r2_scores   = cv_results["test_r2"]
cv_rmse_scores = -cv_results["test_rmse"]
cv_mae_scores  = -cv_results["test_mae"]

print(f"Cross-Validation R² Scores:   {cv_r2_scores}")
print(f"Mean CV R²:   {cv_r2_scores.mean():.6f} (+/- {cv_r2_scores.std() * 2:.6f})")
print(f"\nCross-Validation RMSE Scores: {cv_rmse_scores}")
print(f"Mean CV RMSE: {cv_rmse_scores.mean():.6f} (+/- {cv_rmse_scores.std() * 2:.6f})")
print(f"\nCross-Validation MAE Scores:  {cv_mae_scores}")
print(f"Mean CV MAE:  {cv_mae_scores.mean():.6f} (+/- {cv_mae_scores.std() * 2:.6f})")

Cross-Validation R² Scores:   [0.99901049 0.99666949 0.9981021  0.99784285 0.99942996]
Mean CV R²:   0.998211 (+/- 0.001928)

Cross-Validation RMSE Scores: [0.01996561 0.03848939 0.03024319 0.03299267 0.01673553]
Mean CV RMSE: 0.027685 (+/- 0.016271)

Cross-Validation MAE Scores:  [0.00485452 0.01132448 0.00603925 0.00976919 0.00503717]
Mean CV MAE:  0.007405 (+/- 0.005286)


In [11]:
best_tabpfn = tuned_model.best_model_
perm = permutation_importance(
    best_tabpfn,
    X_test,
    y_test,
    n_repeats=10,
    random_state=SEED,
    scoring="neg_root_mean_squared_error",
    n_jobs=n_cpus,
)

perm_df = (
    pd.DataFrame({
        "feature":         features,
        "importance_mean": perm.importances_mean,
        "importance_std":  perm.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)
print("Permutation importance (drop in score when feature is shuffled):")
print(perm_df)

os.makedirs(PLOT_FOLDER, exist_ok=True)
perm_df.to_csv(os.path.join(PLOT_FOLDER, f"TabPFN_{target}_permutation_importance.csv"), index=False)

X_explain = X_test

shap_values = get_shap_values(
    estimator=best_tabpfn,
    test_x=X_explain,
    attribute_names=features,
)

shap_arr = shap_values.values if hasattr(shap_values, "values") else np.asarray(shap_values)
if shap_arr.ndim == 3:
    shap_arr = shap_arr[:, :, 0]
mean_abs = np.abs(shap_arr).mean(axis=0)

shap_rank_df = (
    pd.DataFrame({"feature": features, "mean_abs_shap": mean_abs})
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)
shap_rank_df.to_csv(os.path.join(PLOT_FOLDER, f"TabPFN_{target}_shap_importance.csv"), index=False)
print("Mean |SHAP| ranking:")
print(shap_rank_df)

# Save artifacts for the plotting notebook
joblib.dump(shap_values, os.path.join(PLOT_FOLDER, f"TabPFN_{target}_shap_values.joblib"))
X_explain.to_csv(os.path.join(PLOT_FOLDER, f"TabPFN_{target}_X_explain.csv"), index=True)
print(f"SHAP values + X_explain saved to {PLOT_FOLDER}/")

metrics["permutation_importance"] = perm_df.to_dict(orient="records")
metrics["shap_importance"]        = shap_rank_df.to_dict(orient="records")

/gpfs/home6/draju/A6/.A6/lib/python3.13/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but TabPFNRegressor was fitted without feature names
  warnings.warn(


/gpfs/home6/draju/A6/.A6/lib/python3.13/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but TabPFNRegressor was fitted without feature names
  warnings.warn(
/gpfs/home6/draju/A6/.A6/lib/python3.13/site-packages/tabpfn/inference.py:1309: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  X_train = torch.as_tensor(X_train, dtype=dtype, device=device)


/gpfs/home6/draju/A6/.A6/lib/python3.13/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but TabPFNRegressor was fitted without feature names
  warnings.warn(
/gpfs/home6/draju/A6/.A6/lib/python3.13/site-packages/tabpfn/inference.py:1309: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  X_train = torch.as_tensor(X_train, dtype=dtype, device=device)


/gpfs/home6/draju/A6/.A6/lib/python3.13/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but TabPFNRegressor was fitted without feature names
  warnings.warn(
/gpfs/home6/draju/A6/.A6/lib/python3.13/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but TabPFNRegressor was fitted without feature names
  warnings.warn(
/gpfs/home6/draju/A6/.A6/lib/python3.13/site-packages/tabpfn/inference.py:1309: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  X_train = torch.as_tensor(X_train, dtype=dtype, device=device)
/gpfs/home6/draju/A6/.A6/lib/python3.13/site-packa

/gpfs/home6/draju/A6/.A6/lib/python3.13/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but TabPFNRegressor was fitted without feature names
  warnings.warn(
/gpfs/home6/draju/A6/.A6/lib/python3.13/site-packages/tabpfn/inference.py:1309: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  X_train = torch.as_tensor(X_train, dtype=dtype, device=device)
/gpfs/home6/draju/A6/.A6/lib/python3.13/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but TabPFNRegressor was fitted without feature names
  warnings.warn(
/gpfs/home6/draju/A6/.A6/lib/python3.13/site-packa

...

ExactExplainer explainer: 100%|█████████▉| 2903/2904 [48:26<00:00,  1.00it/s]

ExactExplainer explainer: 100%|██████████| 2904/2904 [48:27<00:00,  1.01it/s]

ExactExplainer explainer: 2905it [48:28,  1.01it/s]                          

ExactExplainer explainer: 2905it [48:28,  1.00s/it]

Mean |SHAP| ranking:
              feature  mean_abs_shap
0         temperature       0.424315
1            pressure       0.156042
2          z_hydrogen       0.042441
3    z_carbon dioxide       0.040705
4  z_hydrogen sulfide       0.025183
5           z_methane       0.021629
6            z_oxygen       0.019142
7          z_nitrogen       0.015139
8   z_carbon monoxide       0.010157
9             z_argon       0.010135
SHAP values + X_explain saved to /gpfs/home6/draju/A6/TabPFN/with_HPO/SLURMTHICKNESS/


In [12]:
metrics["cv_r2_scores"]    = cv_r2_scores.tolist()
metrics["cv_r2_mean"]      = float(cv_r2_scores.mean())
metrics["cv_r2_std"]       = float(cv_r2_scores.std())
metrics["cv_rmse_scores"]  = cv_rmse_scores.tolist()
metrics["cv_rmse_mean"]    = float(cv_rmse_scores.mean())
metrics["cv_rmse_std"]     = float(cv_rmse_scores.std())
metrics["cv_mae_scores"]   = cv_mae_scores.tolist()
metrics["cv_mae_mean"]     = float(cv_mae_scores.mean())
metrics["cv_mae_std"]      = float(cv_mae_scores.std())
metrics["model"]           = "TabPFN"
metrics["features"]        = features
metrics["target"]          = target
metrics["seed"]            = SEED
metrics["best_hyperparameters"] = {
    k: (v.item() if hasattr(v, "item") else v)
    for k, v in tuned_model.best_params_.items()
}
metrics["best_hpo_score"] = float(tuned_model.best_score_)
metrics["hpo_metric"]     = tuned_model.metric.value
metrics["hpo_n_trials"]   = tuned_model.n_trials

os.makedirs(PLOT_FOLDER, exist_ok=True)
metrics_path = os.path.join(PLOT_FOLDER, f"TabPFN_{target}_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2, default=str)
print(f"\nMetrics saved to: {metrics_path}")


Metrics saved to: /gpfs/home6/draju/A6/TabPFN/with_HPO/SLURMTHICKNESS/TabPFN_interfacial_thickness_metrics.json


In [13]:
notebook_end = time.perf_counter()
elapsed_minutes = (notebook_end - notebook_start) / 60
print(f"Total notebook runtime: {elapsed_minutes:.2f} minutes")

Total notebook runtime: 60.13 minutes
